# Explicit User Invocation for Mode Switching

Explicit user invocation is how power users and developers take direct control over agent behavior. Rather than waiting for the system to infer the right mode from request phrasing, the user issues an explicit command - `/research`, "switch to planning mode", or a structured API parameter. The agent's job is to parse that intent, check whether the user is authorized for that mode, apply any safety confirmation requirements, and switch.

The pattern appears in every major developer-facing product: VS Code Copilot's `/fix` and `/explain` commands, Slack's bot slash commands, Claude Code's own `/commands` palette, and ChatGPT's custom GPT mode selection. It works precisely because the user already knows what they want - the agent should honor that intent exactly rather than second-guessing it.

| Aspect | Explicit invocation | LLM-driven selection |
|--------|--------------------|--------------------|
| Who decides the mode | The user, directly | The classifier LLM |
| Input form | `/command`, NL phrase, UI element | Any user message |
| Latency overhead | Near zero (regex/pattern) | One extra LLM call |
| Error type | Wrong command syntax | Wrong intent inference |
| Best for | Power users, audit-critical systems | General-purpose assistants |
| Cost | Free (no API call needed) | One classifier call per request |

This notebook builds the pattern component by component: slash command parsing, natural language detection, role-based access control, a confirmation gate for high-commitment modes, and a simple conversational loop that wires all four together.

In [1]:
import os
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

### Initializing the LLMs

In [2]:
# Single LLM instance — used for NL mode detection fallback and response generation
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.3)
print("LLM initialized:", llm.model_name)

LLM initialized: gpt-4o-mini


## Mode registry
Before building the command parser, we define the mode registry - the vocabulary of modes users can invoke. Each entry maps a mode name to a system prompt: the behavioral contract the LLM operates under when that mode is active. Two companion structures complete the registry: a permission table specifying which user roles may activate each mode, and a set of modes that require explicit user confirmation before activation.

Behavioral modes (how the agent reasons and presents information) are kept separate from autonomy modes (how much human oversight the agent operates under). Autonomy modes - supervised, semi-autonomous, fully autonomous - are covered in the `02_Autonomy-Modes` section of this repository. Mixing them here would conflate two orthogonal concerns: a research mode governs communication style, while a supervised autonomy level governs approval requirements. Both can apply simultaneously and independently.

In [3]:
# System prompts define agent behavior when a mode is active — injected per call, not stored in history
MODE_PROMPTS: dict[str, str] = {
    "chat": (
        "You are a friendly, knowledgeable assistant. "
        "Keep responses warm, concise, and jargon-free."
    ),
    "research": (
        "You are a rigorous research assistant. "
        "Structure responses with clear headings. "
        "Surface trade-offs, competing perspectives, and confidence levels for key claims."
    ),
    "planning": (
        "You are a structured planning specialist. "
        "Decompose goals into numbered, ordered steps with explicit dependencies and risks. "
        "State assumptions at the top before the plan."
    ),
    "task_execution": (
        "You are a precise execution assistant. "
        "Produce the concrete deliverable requested — code, draft, or analysis — without preamble. "
        "List assumptions at the top, then deliver the output directly."
    ),
    "critic": (
        "You are a constructive critic. "
        "Identify specific weaknesses with exact locations. "
        "For each weakness, propose a concrete improvement. "
        "Score overall quality 1–10 with justification."
    ),
}

# Role-based permission table — which roles may activate each mode
# Principle of least privilege: guest can only use chat; admin has full access
MODE_PERMISSIONS: dict[str, list[str]] = {
    "chat":           ["guest", "user", "admin"],
    "research":       ["user", "admin"],
    "planning":       ["user", "admin"],
    "task_execution": ["user", "admin"],
    "critic":         ["user", "admin"],
}

# Modes that require user confirmation before activation — task_execution takes real actions
# confirming prevents accidental invocation (e.g. /task when /plan was intended)
CONFIRMATION_REQUIRED: set[str] = {"task_execution"}

print(f"Modes registered: {list(MODE_PROMPTS)}")
print(f"Confirmation required for: {CONFIRMATION_REQUIRED}")

Modes registered: ['chat', 'research', 'planning', 'task_execution', 'critic']
Confirmation required for: {'task_execution'}


## Slash command parsing
Slash commands are the clearest interface for explicit mode invocation - unambiguous by design. A message that starts with `/` can only be a command, never content. This makes parsing fast (no LLM call needed) and error messages easy to write: the user clearly intended a command, they just got the syntax wrong.

We support two syntactic forms. The shorthand form maps common aliases to mode names: `/plan` activates planning mode, `/task` activates task_execution. The generic form accepts any mode name: `/mode research`, `/mode critic`. Both forms validate against the mode registry and return a structured result rather than raising exceptions, so the calling code can surface parse errors as helpful messages rather than crashing.

In [4]:
# Short aliases that expand to mode names — cover the most commonly invoked modes
SLASH_ALIASES: dict[str, str] = {
    "/chat":     "chat",
    "/research": "research",
    "/plan":     "planning",
    "/task":     "task_execution",
    "/critic":   "critic",
}


@dataclass
class ParseResult:
    """Result of parsing a potential slash command.

    Attributes:
        is_command: True if the input started with '/'.
        mode: The resolved mode name on success, None otherwise.
        error: Human-readable error message if parsing failed, else empty string.
    """
    is_command: bool
    mode: Optional[str] = None
    error: str = ""


def parse_slash_command(text: str) -> ParseResult:
    """Parse a user message for a slash mode-switch command.

    Handles shorthand aliases (/plan, /research) and the generic /mode <name> form.
    Validates mode names against the MODE_PROMPTS registry.

    Args:
        text: Raw user input, which may or may not start with '/'.

    Returns:
        ParseResult — is_command=False for regular messages, error set on bad syntax.
    """
    stripped = text.strip()

    # Not a command at all — early exit so the caller can take the regular message path
    if not stripped.startswith("/"):
        return ParseResult(is_command=False)

    parts = stripped.lower().split()
    command = parts[0]

    # Check shorthand aliases first — /plan, /research, /critic, etc.
    if command in SLASH_ALIASES:
        return ParseResult(is_command=True, mode=SLASH_ALIASES[command])

    # Handle the generic /mode <name> form
    if command == "/mode":
        if len(parts) < 2:
            # User typed '/mode' with no argument — surface a usage hint
            return ParseResult(
                is_command=True,
                error=f"Usage: /mode <name>. Available modes: {', '.join(sorted(MODE_PROMPTS))}",
            )
        requested = parts[1]
        if requested not in MODE_PROMPTS:
            # Mode name not in registry — surface available options
            return ParseResult(
                is_command=True,
                error=f"Unknown mode '{requested}'. Available: {', '.join(sorted(MODE_PROMPTS))}",
            )
        return ParseResult(is_command=True, mode=requested)

    # Unrecognized slash — typo or unsupported command
    return ParseResult(
        is_command=True,
        error=(
            f"Unknown command '{command}'. "
            f"Try /mode <name> or a shorthand: {', '.join(sorted(SLASH_ALIASES))}"
        ),
    )


print("parse_slash_command defined")
print(f"Shorthand aliases: {list(SLASH_ALIASES)}")

parse_slash_command defined
Shorthand aliases: ['/chat', '/research', '/plan', '/task', '/critic']


`ParseResult` carries three fields. `is_command` tells the caller whether this turn should be treated as a mode switch attempt at all - `False` means the message is regular content. `mode` holds the resolved name on success. `error` holds a human-readable message on failure that the agent can relay directly to the user rather than generating a generic error.

Validating mode names against `MODE_PROMPTS` means adding a new mode to the registry automatically makes it accessible via `/mode <name>`. The shorthand aliases in `SLASH_ALIASES` require explicit entries - intentional friction that keeps the alias table small and curated, rather than auto-generating aliases that users might not expect.

In [5]:
test_inputs = [
    "/research",               # shorthand alias
    "/plan",                   # shorthand alias
    "/mode critic",            # generic form — valid
    "/mode chat",              # generic form — valid
    "/mode unknown_mode",      # bad mode name → error
    "/mode",                   # missing argument → error
    "/bogus_command",          # unrecognized command → error
    "just a regular message",  # not a command at all
    "  /research  ",           # whitespace is stripped
]

print(f"{'Input':<30}  {'Command?':<10} {'Mode':<16}  Result")
print("-" * 82)
for raw in test_inputs:
    r = parse_slash_command(raw)
    mode_col = r.mode or "—"
    # Show the error on failure, "OK" on success, or "not a command" for non-commands
    outcome = r.error if r.error else ("OK" if r.mode else "not a command")
    print(f"{raw!r:<30}  {str(r.is_command):<10} {mode_col:<16}  {outcome}")

Input                           Command?   Mode              Result
----------------------------------------------------------------------------------
'/research'                     True       research          OK
'/plan'                         True       planning          OK
'/mode critic'                  True       critic            OK
'/mode chat'                    True       chat              OK
'/mode unknown_mode'            True       —                 Unknown mode 'unknown_mode'. Available: chat, critic, planning, research, task_execution
'/mode'                         True       —                 Usage: /mode <name>. Available modes: chat, critic, planning, research, task_execution
'/bogus_command'                True       —                 Unknown command '/bogus_command'. Try /mode <name> or a shorthand: /chat, /critic, /plan, /research, /task
'just a regular message'        False      —                 not a command
'  /research  '                 True       research 

## Natural language detection
Slash commands work well for developer-facing interfaces, but consumer applications and voice interfaces need to handle natural language mode requests: "switch to research mode", "can you be more critical of this?", "just do it". The user won't type `/mode` - they'll express their intent as a sentence.

The detection strategy uses two stages. First, a fast pattern match checks the message against a dictionary of known phrases. This handles the common, unambiguous cases - "switch to research mode" cannot mean anything other than a mode switch request - without any LLM call. Second, if the message contains mode-related keywords but doesn't match any known pattern, an LLM classifier resolves the ambiguity. The LLM fallback fires rarely in practice: most natural language mode requests use one of a small set of standard phrasings. Keeping the LLM as a fallback rather than the primary path means the common case costs nothing.

In [6]:
# Known phrase patterns matched before any LLM call — cover common unambiguous requests
NL_PATTERNS: dict[str, list[str]] = {
    "chat": [
        "switch to chat mode", "casual mode", "just chat", "friendly mode", "relax mode",
    ],
    "research": [
        "switch to research mode", "research mode", "go into research mode", "activate research mode", "enter research mode",
    ],
    "planning": [
        "switch to planning mode", "planning mode", "go into planning mode", "help me plan", "planning mode please", "enter planning mode",
    ],
    "task_execution": [
        "just do it", "go ahead and do it", "execute this", "switch to task mode", "task execution mode", "task mode",
    ],
    "critic": [
        "critic mode", "switch to critic mode", "activate critic mode", "critique this", "give me feedback on", "review this critically",
    ],
}

# Words that suggest mode-related intent — used as a cheap pre-filter before the LLM call
# Avoids sending every regular message to the classifier
MODE_KEYWORDS = [
    "mode", "switch", "change", "activate", "research", "planning", "critic", "critique", "execute", "task", "casual", "friendly",
]


def detect_nl_mode_request(text: str) -> tuple[bool, Optional[str], str]:
    """Detect an explicit mode-switch request in a natural language message.

    Two-stage: pattern match first (free), LLM fallback only when keywords are present but no phrase matches. Returns the detection source for monitoring.

    Args:
        text: The user's raw input.

    Returns:
        (mode_requested, mode_name_or_None, source)
        source is 'pattern' for pattern match, 'llm' for LLM fallback, 'none' otherwise.
    """
    lower = text.lower()

    # Stage 1: scan known phrases — no API call needed
    for mode, phrases in NL_PATTERNS.items():
        if any(phrase in lower for phrase in phrases):
            return True, mode, "pattern"

    # Stage 2: LLM fallback — only when mode-related keywords are present
    if any(kw in lower for kw in MODE_KEYWORDS):
        return _llm_detect(text)

    # No keywords present — definitely not a mode request
    return False, None, "none"


def _llm_detect(text: str) -> tuple[bool, Optional[str], str]:
    """Use the LLM to classify an ambiguous potential mode-switch message.

    Called only by detect_nl_mode_request when pattern matching finds keywords but no phrase.

    Args:
        text: The user message to classify.

    Returns:
        (mode_requested, mode_name_or_None, 'llm')
    """
    modes_list = ", ".join(sorted(MODE_PROMPTS))
    # Single-line response format avoids JSON parsing overhead
    prompt = (
        f"You are a mode-switch classifier. Determine if the user wants to change the agent's mode.\n"
        f"Available modes: {modes_list}\n"
        f"User message: \"{text}\"\n\n"
        f"Reply with EXACTLY one line: YES <mode_name>  or  NO\n"
        f"Only reply YES if the user is clearly requesting a mode change."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()

    if content.upper().startswith("YES"):
        # Parse the mode name from the response (e.g. "YES research" → "research")
        parts = content.split()
        if len(parts) >= 2 and parts[1].lower() in MODE_PROMPTS:
            return True, parts[1].lower(), "llm"

    # Either NO or a malformed response — treat as not a mode request
    return False, None, "llm"


print("detect_nl_mode_request defined")
print(f"Pattern-matched modes: {list(NL_PATTERNS)}")

detect_nl_mode_request defined
Pattern-matched modes: ['chat', 'research', 'planning', 'task_execution', 'critic']


The two-stage design keeps the common case free. Pattern matching handles unambiguous requests - "switch to research mode", "critique this" - with zero API cost. The LLM fallback fires only when mode-related keywords are present but no phrase matches, covering paraphrase and novel phrasing that the pattern list doesn't anticipate. The `source` return value makes it possible to monitor the ratio of pattern hits to LLM calls in production - a high LLM ratio signals that the pattern list is missing common phrases.

The LLM prompt asks for a single-line response (`YES <mode_name>` or `NO`) rather than JSON, which is cheaper, faster to generate, and requires no parsing library - just splitting on whitespace.

In [7]:
nl_test_cases = [
    ("Switch to research mode please.",                       "pattern → research"),
    ("Let's go into planning mode.",                          "pattern → planning"),
    ("Critique this proposal for me.",                        "pattern → critic"),
    ("Just do it — go ahead.",                                "pattern → task_execution"),
    ("I'd like a different kind of response style.",          "keyword → LLM fallback"),
    ("What is the capital of France?",                        "no keywords → none"),
    ("Can you help me think through this more carefully?",    "no mode keywords → none"),
]

print(f"{'Message':<50}  {'Requested?':<12} {'Mode':<16}  Source  (expected)")
print("-" * 100)
for msg, expected_label in nl_test_cases:
    requested, mode, source = detect_nl_mode_request(msg)
    short = msg[:48] + ".." if len(msg) > 50 else msg
    mode_col = mode or "—"
    print(f"{short:<50}  {str(requested):<12} {mode_col:<16}  {source:<8}  ({expected_label})")

Message                                             Requested?   Mode              Source  (expected)
----------------------------------------------------------------------------------------------------
Switch to research mode please.                     True         research          pattern   (pattern → research)
Let's go into planning mode.                        True         planning          pattern   (pattern → planning)
Critique this proposal for me.                      True         critic            pattern   (pattern → critic)
Just do it — go ahead.                              True         task_execution    pattern   (pattern → task_execution)
I'd like a different kind of response style.        False        —                 none      (keyword → LLM fallback)
What is the capital of France?                      False        —                 none      (no keywords → none)
Can you help me think through this more carefully?  False        —                 none      (no mode key

## Role-based access control
With parsing complete, the next question is authorization: should this user be allowed to activate the requested mode? In multi-user systems or any application with role tiers, different users should have access to different capabilities. A guest user browsing a product assistant should not be able to activate `task_execution` mode and trigger the agent to take real actions.

The permission check is a dictionary lookup. `MODE_PERMISSIONS` maps each mode to the list of roles permitted to use it, following the principle of least privilege: start with the minimum access and grant upward explicitly. The function returns a tuple rather than raising an exception, so the calling code can compose it cleanly with the confirmation gate and state updates without try-except blocks.

In [8]:
def check_permission(mode: str, user_role: str) -> tuple[bool, str]:
    """Check whether a user role is authorized to activate a mode.

    Reads directly from MODE_PERMISSIONS. Returns a tuple so the caller can handle both outcomes without exception handling.

    Args:
        mode: The mode the user wants to activate.
        user_role: The user's role string (e.g. 'guest', 'user', 'admin').

    Returns:
        (authorized, message) — message explains both approvals and denials.
    """
    allowed_roles = MODE_PERMISSIONS.get(mode, [])

    if user_role in allowed_roles:
        return True, f"Role '{user_role}' authorized for '{mode}' mode."

    # Include the required roles in the denial message so users know what they need
    return False, (
        f"Role '{user_role}' cannot activate '{mode}' mode. "
        f"Required: one of {allowed_roles}."
    )


def accessible_modes(user_role: str) -> list[str]:
    """Return all modes a role is authorized to use — useful for help messages and UI menus.

    Args:
        user_role: The user's role string.

    Returns:
        Sorted list of mode names the role may activate.
    """
    return sorted(
        mode for mode, roles in MODE_PERMISSIONS.items() if user_role in roles
    )


# Show what each role tier can access
for role in ["guest", "user", "admin"]:
    print(f"  {role:<8}: {accessible_modes(role)}")

  guest   : ['chat']
  user    : ['chat', 'critic', 'planning', 'research', 'task_execution']
  admin   : ['chat', 'critic', 'planning', 'research', 'task_execution']


`check_permission` and `accessible_modes` read from the same `MODE_PERMISSIONS` table and serve different callers. The former answers "can this user do this specific thing right now?" - a per-request check called on every mode switch attempt. The latter answers "what can this user do?" - called once to populate a help message or render a UI dropdown showing only permitted modes.

The denial message names the required roles rather than just saying "access denied". This gives the user actionable information: they know exactly which role tier would grant access, without the agent exposing internal permission logic or revealing what other users can do.

In [9]:
# Permission matrix — representative combinations across roles and modes
attempts = [
    ("guest",  "chat"),             # allowed — guest baseline
    ("guest",  "research"),         # denied — guest cannot research
    ("guest",  "task_execution"),   # denied — guest cannot execute tasks
    ("user",   "research"),         # allowed
    ("user",   "task_execution"),   # allowed
    ("user",   "critic"),           # allowed
    ("admin",  "task_execution"),   # allowed
    ("admin",  "critic"),           # allowed
]

print(f"{'Role':<8} {'Mode':<16}  {'Auth?':<8}  Message")
print("-" * 85)
for role, mode in attempts:
    authorized, message = check_permission(mode, role)
    status = "ALLOWED" if authorized else "DENIED "
    print(f"{role:<8} {mode:<16}  [{status}]  {message}")

Role     Mode              Auth?     Message
-------------------------------------------------------------------------------------
guest    chat              [ALLOWED]  Role 'guest' authorized for 'chat' mode.
guest    research          [DENIED ]  Role 'guest' cannot activate 'research' mode. Required: one of ['user', 'admin'].
guest    task_execution    [DENIED ]  Role 'guest' cannot activate 'task_execution' mode. Required: one of ['user', 'admin'].
user     research          [ALLOWED]  Role 'user' authorized for 'research' mode.
user     task_execution    [ALLOWED]  Role 'user' authorized for 'task_execution' mode.
user     critic            [ALLOWED]  Role 'user' authorized for 'critic' mode.
admin    task_execution    [ALLOWED]  Role 'admin' authorized for 'task_execution' mode.
admin    critic            [ALLOWED]  Role 'admin' authorized for 'critic' mode.


## Confirmation gate
Authorization answers whether a user *can* switch to a mode. The confirmation gate answers a different question: did this user actually *mean* to activate this mode, or did they trigger it accidentally? For modes that take real actions or produce outputs that are hard to revise, requiring a one-sentence acknowledgement before activation converts accidental invocations into deliberate ones.

`task_execution` is the natural candidate here - it instructs the agent to produce concrete deliverables and proceed without asking for guidance at each step. A user who typed `/task` when they meant `/plan` gets a chance to reconsider before the agent dives into execution. The gate is intentionally lightweight: one sentence describing the mode's implications, plus a yes/no prompt. It is not a detailed safety review; for that, see the autonomy confirmation pattern in `02_Autonomy-Modes`.

In [10]:
# Implications shown to users when confirmation is required — describes what the mode means to them
# Intentionally separate from MODE_PROMPTS (which describe what the mode means to the LLM)
MODE_IMPLICATIONS: dict[str, str] = {
    "task_execution": (
        "Task execution mode produces concrete outputs and takes real actions. "
        "I will proceed directly without asking for guidance at each step."
    ),
}


def build_confirmation_message(mode: str, current_mode: str) -> str:
    """Build the confirmation prompt shown before activating a high-commitment mode.

    Returns an empty string if no confirmation is required for this mode, so the caller only needs 'if msg' rather than a separate boolean flag.

    Args:
        mode: The mode the user wants to activate.
        current_mode: The mode currently active in the session.

    Returns:
        Confirmation prompt string, or '' if mode does not require confirmation.
    """
    if mode not in CONFIRMATION_REQUIRED:
        # No confirmation needed — empty string signals immediate switch is safe
        return ""

    implications = MODE_IMPLICATIONS.get(
        mode, f"'{mode}' has significant behavioral implications."
    )
    return (
        f"You're requesting to switch from '{current_mode}' to '{mode}'.\n"
        f"{implications}\n\n"
        f"Reply 'yes' to confirm, or 'no' to stay in '{current_mode}' mode."
    )


print(f"Confirmation required for: {CONFIRMATION_REQUIRED}")
print(f"Immediate switch (no confirmation): {sorted(set(MODE_PROMPTS) - CONFIRMATION_REQUIRED)}")

Confirmation required for: {'task_execution'}
Immediate switch (no confirmation): ['chat', 'critic', 'planning', 'research']


In [11]:
# Show the confirmation message for each mode — empty string means immediate switch
print("CONFIRMATION GATE BEHAVIOR")
print("-" * 60)
for mode in sorted(MODE_PROMPTS):
    msg = build_confirmation_message(mode, current_mode="chat")
    if msg:
        print(f"\n/mode {mode}  →  confirmation required:")
        # Indent each line of the message for readable display
        for line in msg.split("\n"):
            print(f"  {line}")
    else:
        print(f"/mode {mode}  →  immediate switch (no confirmation)")

CONFIRMATION GATE BEHAVIOR
------------------------------------------------------------
/mode chat  →  immediate switch (no confirmation)
/mode critic  →  immediate switch (no confirmation)
/mode planning  →  immediate switch (no confirmation)
/mode research  →  immediate switch (no confirmation)

/mode task_execution  →  confirmation required:
  You're requesting to switch from 'chat' to 'task_execution'.
  Task execution mode produces concrete outputs and takes real actions. I will proceed directly without asking for guidance at each step.
  
  Reply 'yes' to confirm, or 'no' to stay in 'chat' mode.


`build_confirmation_message` returns an empty string when no confirmation is needed, letting the caller use `if msg` rather than a separate boolean. The `MODE_IMPLICATIONS` dictionary is separate from `MODE_PROMPTS` - implications describe what the mode means to the *user* ("I will proceed without asking"), while system prompts describe what the mode means to the *LLM* (behavioral instructions about tone, format, and output structure).

## Agent session and control flow
With the four components defined - slash parsing, natural language detection, permission checks, and confirmation - we can assemble them into a simple conversational agent. The agent maintains session state across turns: the active mode, the user's role, any pending confirmation, and an audit log of every mode switch attempt.

`AgentSession` is a plain dataclass rather than a LangGraph state. Explicit invocation follows a fixed control flow (parse → check permission → confirm if needed → respond) that does not require graph routing. A simple session object and a `run_turn` function express this clearly with considerably less overhead than a StateGraph. Reserve graph frameworks for workflows with complex branching or parallel execution - this isn't one of them.

In [12]:
@dataclass
class AgentSession:
    """Mutable session state for the explicit invocation agent.

    Attributes:
        current_mode: The active behavioral mode name.
        user_role: The user's role string, checked on every mode switch attempt.
        pending_mode: Mode awaiting confirmation — set when confirmation is requested,
                      cleared when the user replies yes or no.
        history: Conversation history (user + assistant messages only; system prompts
                 are injected fresh per call and never stored here).
        audit_log: Record of every mode switch attempt in this session.
    """
    current_mode: str           = "chat"
    user_role:    str           = "user"
    pending_mode: Optional[str] = None
    # field(default_factory=list) is required for mutable defaults in dataclasses
    history:      list          = field(default_factory=list)
    audit_log:    list          = field(default_factory=list)


def new_session(user_role: str = "user", initial_mode: str = "chat") -> AgentSession:
    """Create a fresh session for a user.

    Args:
        user_role: The user's role ('guest', 'user', 'admin').
        initial_mode: Starting mode — defaults to 'chat'.

    Returns:
        An initialized AgentSession with empty history and audit log.
    """
    return AgentSession(user_role=user_role, current_mode=initial_mode)


print("AgentSession defined — fields:", list(AgentSession.__dataclass_fields__))

AgentSession defined — fields: ['current_mode', 'user_role', 'pending_mode', 'history', 'audit_log']


`run_turn` processes one user turn through the full explicit invocation pipeline. The order of checks matters. Pending confirmations resolve first - if the agent asked the user "are you sure?", the next turn must check for yes/no before treating the message as a new command. Then slash commands take priority over natural language detection. Regular messages fall through to response generation.

The response step injects the current mode's system prompt fresh for each LLM call. System prompts are deliberately not stored in `history` - persisting them would create stale mode-system-prompt artifacts in the conversation record when the mode changes, which would confuse both the LLM and any downstream inspection of the history.

In [13]:
def _log_switch(session: AgentSession, method: str, requested: str, authorized: bool, note: str = "") -> None:
    """Append a mode switch attempt to the session audit log.

    Args:
        session: The session to update.
        method: How the switch was requested ('slash', 'natural_language', 'confirmation').
        requested: The mode name the user requested.
        authorized: Whether the switch was permitted.
        note: Optional context — denial reason, 'awaiting confirmation', etc.
    """
    session.audit_log.append({
        "ts":         datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "user_role":  session.user_role,
        "method":     method,
        "from_mode":  session.current_mode,
        "to_mode":    requested,
        "authorized": authorized,
        "note":       note,
    })


def _generate_response(session: AgentSession) -> str:
    """Generate a response in the current mode using the full conversation history.

    Injects the mode system prompt fresh for this call — never persists it to history.

    Args:
        session: Current session (current_mode determines the system prompt).

    Returns:
        The LLM's response string.
    """
    system_prompt = MODE_PROMPTS.get(session.current_mode, MODE_PROMPTS["chat"])
    # Build the message list: [mode system prompt] + [full conversation history]
    messages = [SystemMessage(content=system_prompt)] + session.history
    response = llm.invoke(messages)
    # Append the assistant response to history so it is available on the next turn
    session.history.append(AIMessage(content=response.content))
    return response.content


def _handle_mode_request(session: AgentSession, requested_mode: str, method: str) -> str:
    """Apply the full mode-switch pipeline: permission → confirmation → switch.

    Args:
        session: Current session state (mutated in place on successful switch).
        requested_mode: The mode the user wants to activate.
        method: 'slash' or 'natural_language'.

    Returns:
        Agent response string.
    """
    # Step 1: permission check — does this role have access to the requested mode?
    authorized, auth_message = check_permission(requested_mode, session.user_role)
    if not authorized:
        _log_switch(session, method, requested_mode, False, note=auth_message)
        response_text = f"Mode switch denied. {auth_message}"
        session.history.append(AIMessage(content=response_text))
        return response_text

    # Step 2: confirmation gate — does this mode require an explicit acknowledgement?
    confirm_msg = build_confirmation_message(requested_mode, session.current_mode)
    if confirm_msg:
        # Store the pending mode and send the confirmation prompt — resolved on the next turn
        session.pending_mode = requested_mode
        _log_switch(session, method, requested_mode, True, note="awaiting confirmation")
        session.history.append(AIMessage(content=confirm_msg))
        return confirm_msg

    # Step 3: all clear — apply the switch immediately
    _log_switch(session, method, requested_mode, True)
    previous = session.current_mode
    session.current_mode = requested_mode
    response_text = f"Switched from '{previous}' to '{requested_mode}' mode. What would you like to do?"
    session.history.append(AIMessage(content=response_text))
    return response_text


def run_turn(session: AgentSession, user_text: str) -> str:
    """Process one user turn through the explicit invocation pipeline.

    Priority order: pending confirmation → slash command → NL detection → regular message.
    Mutates session state in place.

    Args:
        session: Current session state (mutated in place).
        user_text: The user's raw input for this turn.

    Returns:
        The agent's response string.
    """
    # Add the user message to history immediately so it appears in subsequent LLM calls
    session.history.append(HumanMessage(content=user_text))
    lower = user_text.strip().lower()

    # --- Priority 1: resolve any pending confirmation (user replying yes/no) ---
    if session.pending_mode:
        if lower in ("yes", "y"):
            confirmed_mode = session.pending_mode
            session.pending_mode = None
            _log_switch(session, "confirmation", confirmed_mode, True)
            session.current_mode = confirmed_mode
            response_text = f"Switched to '{confirmed_mode}' mode. How can I help?"
            session.history.append(AIMessage(content=response_text))
            return response_text

        elif lower in ("no", "n"):
            cancelled = session.pending_mode
            session.pending_mode = None
            _log_switch(session, "confirmation", cancelled, False, note="user cancelled")
            response_text = f"Understood. Staying in '{session.current_mode}' mode."
            session.history.append(AIMessage(content=response_text))
            return response_text
        # If neither yes nor no, fall through and treat the message normally

    # --- Priority 2: slash command ---
    slash_result = parse_slash_command(user_text)
    if slash_result.is_command:
        if slash_result.error:
            # Parse error — surface the error message to the user
            response_text = f"Command error: {slash_result.error}"
            session.history.append(AIMessage(content=response_text))
            return response_text
        return _handle_mode_request(session, slash_result.mode, method="slash")

    # --- Priority 3: natural language mode request ---
    nl_requested, nl_mode, _ = detect_nl_mode_request(user_text)
    if nl_requested and nl_mode:
        return _handle_mode_request(session, nl_mode, method="natural_language")

    # --- Priority 4: regular message — generate a response in the current mode ---
    return _generate_response(session)


print("run_turn and helper functions defined")

run_turn and helper functions defined


## Audit log
Every explicit mode invocation - successful or denied - has been written to the session audit log throughout each demo run. The log captures the who, when, how, what, and whether for every mode switch attempt, giving a complete picture of mode governance across a session.

In production, these entries would stream to an observability platform (Datadog, Splunk, a purpose-built audit database) rather than staying in memory. The structure maps cleanly to standard audit log schemas: timestamp, actor, action, target resource, and authorization outcome.

In [14]:
def print_audit_log(session: AgentSession) -> None:
    """Print the session audit log in a readable table format.

    Args:
        session: The session whose audit log to display.
    """
    if not session.audit_log:
        print("  (no mode switch attempts recorded)")
        return

    print(f"  {len(session.audit_log)} entries")
    print(f"  {'Timestamp':<22} {'Role':<8} {'Method':<18} {'From':<16} {'To':<16} Auth   Note")
    print("  " + "-" * 95)
    for e in session.audit_log:
        status = "ALLOWED" if e["authorized"] else "DENIED "
        note = f"  {e['note']}" if e["note"] else ""
        print(
            f"  {e['ts']:<22} {e['user_role']:<8} {e['method']:<18} "
            f"{e['from_mode']:<16} {e['to_mode']:<16} [{status}]{note}"
        )


print("print_audit_log defined")

print_audit_log defined


## Demonstrations

With all components assembled, we run three sessions that cover the full range of behaviors: slash commands, natural language requests, RBAC denials, and the confirmation flow. The sessions use different user roles to show how permission boundaries affect mode access.

In [15]:
print("SESSION 1 — user role: slash commands, NL requests, confirmation flow")
print("=" * 75)

session = new_session(user_role="user", initial_mode="chat")

turns = [
    ("Hello! What can you help me with today?",
     "chat — regular message"),
    ("/research",
     "slash → research (immediate)"),
    ("What are the trade-offs between RAG and fine-tuning for domain adaptation?",
     "response in research mode"),
    ("Switch to planning mode please.",
     "NL → planning (immediate)"),
    ("Create a 3-step plan to implement a semantic search feature.",
     "response in planning mode"),
    ("/task",
     "slash → task_execution (confirmation required)"),
    ("yes",
     "confirmation → task_execution applied"),
    ("Write a Python function that chunks text into 512-token segments.",
     "response in task_execution mode"),
]

for user_input, label in turns:
    print(f"\n  [{session.current_mode.upper():<14}] USER: {user_input}")
    print(f"   ({label})")
    response = run_turn(session, user_input)
    # Truncate long LLM responses — focus is on mode transitions, not response content
    preview = response[:250] + "..." if len(response) > 250 else response
    print(f"   AGENT: {preview}")

SESSION 1 — user role: slash commands, NL requests, confirmation flow

  [CHAT          ] USER: Hello! What can you help me with today?
   (chat — regular message)
   AGENT: Hello! I'm here to help with any questions you have or topics you'd like to discuss. Whether it's information, advice, or just a friendly chat, feel free to ask!

  [CHAT          ] USER: /research
   (slash → research (immediate))
   AGENT: Switched from 'chat' to 'research' mode. What would you like to do?

  [RESEARCH      ] USER: What are the trade-offs between RAG and fine-tuning for domain adaptation?
   (response in research mode)
   AGENT: ### Overview of RAG and Fine-Tuning

**RAG (Retrieval-Augmented Generation)**: This approach combines retrieval mechanisms with generative models. It retrieves relevant documents from a knowledge base and uses them to inform the generation of respons...

  [RESEARCH      ] USER: Switch to planning mode please.
   (NL → planning (immediate))
   AGENT: Switched from 'resear

In [16]:
print("SESSION 2 — guest role: RBAC denials and command error handling")
print("=" * 75)

guest = new_session(user_role="guest", initial_mode="chat")

turns = [
    ("Hi! Can you explain what retrieval-augmented generation is?",
     "chat — allowed for guest"),
    ("/research",
     "slash — denied (guest cannot research)"),
    ("/task",
     "slash — denied (guest cannot execute tasks)"),
    ("/bogus",
     "unrecognized command → parse error"),
    ("/mode",
     "missing argument → parse error"),
    ("/chat",
     "chat → chat (same mode, allowed for guest)"),
]

for user_input, label in turns:
    print(f"\n  [{guest.current_mode.upper():<6}] USER: {user_input}  ({label})")
    response = run_turn(guest, user_input)
    print(f"   AGENT: {response[:200]}{'...' if len(response) > 200 else ''}")

SESSION 2 — guest role: RBAC denials and command error handling

  [CHAT  ] USER: Hi! Can you explain what retrieval-augmented generation is?  (chat — allowed for guest)
   AGENT: Of course! Retrieval-augmented generation (RAG) is a technique that combines two powerful approaches in natural language processing: retrieval and generation.

Here's how it works:

1. **Retrieval**: ...

  [CHAT  ] USER: /research  (slash — denied (guest cannot research))
   AGENT: Mode switch denied. Role 'guest' cannot activate 'research' mode. Required: one of ['user', 'admin'].

  [CHAT  ] USER: /task  (slash — denied (guest cannot execute tasks))
   AGENT: Mode switch denied. Role 'guest' cannot activate 'task_execution' mode. Required: one of ['user', 'admin'].

  [CHAT  ] USER: /bogus  (unrecognized command → parse error)
   AGENT: Command error: Unknown command '/bogus'. Try /mode <name> or a shorthand: /chat, /critic, /plan, /research, /task

  [CHAT  ] USER: /mode  (missing argument → parse error)


In [17]:
print("SESSION 3 — admin role: confirmation cancellation then critic response")
print("=" * 75)

admin = new_session(user_role="admin", initial_mode="chat")

turns = [
    ("/task",
     "task_execution — confirmation required"),
    ("no",
     "cancel → stay in chat"),
    ("/task",
     "task_execution — confirmation required again"),
    ("yes",
     "confirm → task_execution applied"),
    ("/critic",
     "critic — immediate switch"),
    ("Here is my plan: Step 1: build it. Step 2: test it. Step 3: ship it. What is missing?",
     "critic response"),
]

for user_input, label in turns:
    print(f"\n  [{admin.current_mode.upper():<14}] USER: {user_input}")
    print(f"   ({label})")
    response = run_turn(admin, user_input)
    print(f"   AGENT: {response[:300]}{'...' if len(response) > 300 else ''}")

SESSION 3 — admin role: confirmation cancellation then critic response

  [CHAT          ] USER: /task
   (task_execution — confirmation required)
   AGENT: You're requesting to switch from 'chat' to 'task_execution'.
Task execution mode produces concrete outputs and takes real actions. I will proceed directly without asking for guidance at each step.

Reply 'yes' to confirm, or 'no' to stay in 'chat' mode.

  [CHAT          ] USER: no
   (cancel → stay in chat)
   AGENT: Understood. Staying in 'chat' mode.

  [CHAT          ] USER: /task
   (task_execution — confirmation required again)
   AGENT: You're requesting to switch from 'chat' to 'task_execution'.
Task execution mode produces concrete outputs and takes real actions. I will proceed directly without asking for guidance at each step.

Reply 'yes' to confirm, or 'no' to stay in 'chat' mode.

  [CHAT          ] USER: yes
   (confirm → task_execution applied)
   AGENT: Switched to 'task_execution' mode. How can I help?

  [TASK_EXE

## Reviewing the audit logs
The audit logs from all three sessions show every mode switch attempt with full context. In a real deployment these entries would be the primary evidence trail for compliance review and the primary diagnostic tool for investigating unexpected mode activations.

In [18]:
print("=== Session 1 (user role) ===")
print_audit_log(session)

print("\n=== Session 2 (guest role) ===")
print_audit_log(guest)

print("\n=== Session 3 (admin role) ===")
print_audit_log(admin)

=== Session 1 (user role) ===
  4 entries
  Timestamp              Role     Method             From             To               Auth   Note
  -----------------------------------------------------------------------------------------------
  2026-06-04T19:09:44+00:00 user     slash              chat             research         [ALLOWED]
  2026-06-04T19:09:58+00:00 user     natural_language   research         planning         [ALLOWED]
  2026-06-04T19:10:07+00:00 user     slash              planning         task_execution   [ALLOWED]  awaiting confirmation
  2026-06-04T19:10:07+00:00 user     confirmation       planning         task_execution   [ALLOWED]

=== Session 2 (guest role) ===
  3 entries
  Timestamp              Role     Method             From             To               Auth   Note
  -----------------------------------------------------------------------------------------------
  2026-06-04T19:10:15+00:00 guest    slash              chat             research         [DENIED